# **Section 6, Correlation, regression and inference**

Everything runs on its own, the data is built in the
notebook, so you can change a number and re-run.


#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- [15 Prediction](https://inferentialthinking.com/chapters/15/prediction/)
- [16 Inference for Regression](https://inferentialthinking.com/chapters/16/inference-for-regression/)

**Where this is used.** 
- This Section feeds **Lab 5** (correlation, the regression
line, prediction and residuals) and 
- **Lab 6** (RMSE, `minimize`, and the
bootstrap for a slope). Neither lab feeds a project; they are assessed in the
exam.


In [ ]:
# Run this cell first -- the install takes about a minute in the browser
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Correlation](#1)
2. [What r does and does not tell you](#2)
3. [The regression line](#3)
4. [Prediction](#4)
5. [RMSE, how wrong are we?](#5)
6. [`minimize`](#6)
7. [Residual plots](#7)
8. [Inference: is the slope real?](#8)
9. [Quick reference](#9)


---

<a id='1'></a>
## **1. Correlation**

The **correlation coefficient** $r$ measures how tightly two numerical variables
follow a straight line. It runs from −1 to 1.

The definition is one line, once both variables are in standard units:

$$r = \text{mean of } (x \text{ in SU}) \times (y \text{ in SU})$$


In [ ]:
def standard_units(x):
    """Converts an array to standard units."""
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    """The correlation coefficient between two columns of a table."""
    return np.mean(standard_units(t.column(x)) * standard_units(t.column(y)))

### **One thing to watch: these signatures differ in the labs**

The textbook and the lectures write these functions to take a **table and two
column names**. Labs 5 and 6 write them to take **two arrays**.

| here, in the lectures and in the textbook | in Labs 5 and 6 |
|---|---|
| `correlation(t, x, y)` | `correlation(x, y)` |
| `slope(t, x, y)` | `slope(x, y)` |
| `intercept(t, x, y)` | `intercept(x, y)` |

The textbook and lectures use the first; the labs were built from uses the second. They compute exactly the same number.

The difference is only *where the columns come out of the table*. The array
version pulls them out before calling, which is what Lab 5's
`predict(tbl, col1, col2)` does in its first two lines:

```python
x = tbl.column(col1)
y = tbl.column(col2)
```

Use whichever form the question in front of you asks for.

In [ ]:
# Hours of revision and exam mark, 60 students
np.random.seed(6)
hrs = np.round(np.random.uniform(0, 12, 60), 1)
study = Table().with_columns(
    'Hours', hrs,
    'Mark', np.round(38 + 4.1 * hrs + np.random.normal(0, 6, 60), 1).clip(0, 100))
study.scatter('Hours', 'Mark', fit_line=True)
plots.title('r = ' + str(round(correlation(study, 'Hours', 'Mark'), 3)))
plots.show()

Why does that product work? In standard units, a point above both means gives a
positive product, and so does a point below both. Points that go against the
trend give a negative one. Averaging them measures how consistently the two move
together.


---

<a id='2'></a>
## **2. What r does and does not tell you**

$r$ measures **linear** association only. A perfect relationship that is not a
straight line can give $r$ near zero.


In [ ]:
x = np.arange(-4, 4.1, 0.25)
curve = Table().with_columns('x', x, 'y', x ** 2)
curve.scatter('x', 'y')
plots.title('r = ' + str(round(correlation(curve, 'x', 'y'), 3))
            + ' -- but the relationship is exact')
plots.show()

Three things $r$ does not tell you:

- **The slope.** $r$ is unitless; the slope has units.
- **Whether a line is appropriate.** Always look at the scatter first.
- **Causation.** Same argument as Section 4, an association can come from a
  confounder, and no amount of correlation rules that out.

> **Always plot before you compute.** Four datasets can share an identical $r$
> and look completely different. Computing $r$ on data you have not looked at is
> how people report relationships that are not there.


---

<a id='3'></a>
## **3. The regression line**

In standard units the regression line is beautifully simple: it passes through
the origin with slope $r$.

$$\text{predicted } y_{su} = r \times x_{su}$$

Converting back to the original units gives the familiar form:

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}
\qquad
\text{intercept} = \text{mean of } y - \text{slope} \times \text{mean of } x$$


In [ ]:
def slope(t, x, y):
    """The slope of the regression line, in original units."""
    r = correlation(t, x, y)
    return r * np.std(t.column(y)) / np.std(t.column(x))

def intercept(t, x, y):
    """The intercept of the regression line."""
    return np.mean(t.column(y)) - slope(t, x, y) * np.mean(t.column(x))

a = slope(study, 'Hours', 'Mark')
b = intercept(study, 'Hours', 'Mark')
print(f'mark = {a:.4f} x hours + {b:.3f}')

### **Why it is called *regression***

Because the slope is $r$ in standard units and $|r| \le 1$, a prediction is
always **closer to average** than the input was. Someone two SDs above average in
revision is predicted to be less than two SDs above average in mark.

That pull towards the mean is where the name comes from, and it is not a flaw, 
it follows from the line being the best predictor.


---

<a id='4'></a>
## **4. Prediction**


In [ ]:
def fitted(t, x, y, value):
    """Predicts y for a given x, using the regression line."""
    return slope(t, x, y) * value + intercept(t, x, y)

for hgt in [2, 6, 11]:
    print(f'{hgt} hours -> predicted mark {fitted(study, "Hours", "Mark", hgt):.1f}')

Lab 5 calls this `predict`, and it takes the whole column at once rather than a
single value:

```python
predict(tbl, col1, col2)      # an array of predictions, one per row
fitted(t, x, y, value)        # one prediction, for one x
```

> **Do not predict far outside the data.** The line is fitted where the points
> are. Feeding it 40 hours gives a number, and the number means nothing, nobody
> in the data revised anywhere near that long.


---

<a id='5'></a>
## **5. RMSE, how wrong are we?**

A prediction has an **error**: actual minus predicted. Errors are positive and
negative, so squaring before averaging stops them cancelling. The square root
returns the answer to the original units.

$$\text{RMSE} = \sqrt{\text{mean of }(\text{actual} - \text{predicted})^2}$$

Same shape as the standard deviation, and that is not a coincidence.


In [ ]:
def rmse_of_line(any_slope, any_intercept):
    """Root mean squared error of a line on the study data."""
    predicted = any_slope * study.column('Hours') + any_intercept
    return np.sqrt(np.mean((study.column('Mark') - predicted) ** 2))

print(f'the regression line:   {rmse_of_line(a, b):.4f}')
print(f'a slightly wrong line: {rmse_of_line(a + 0.5, b):.4f}')
print(f'a badly wrong line:    {rmse_of_line(1.0, 60):.4f}')

Every alternative is worse. That is the defining property: **the regression line
is the line with the smallest RMSE.** No other slope and intercept beat it.


---

<a id='6'></a>
## **6. `minimize`**

You can find the best line without the formula at all. `minimize` takes a
function and searches numerically for the inputs that make it smallest.

```
minimize(function)      # returns an array of the best arguments
```


In [ ]:
best = minimize(rmse_of_line)
print('minimize says: slope', round(best.item(0), 4),
      ' intercept', round(best.item(1), 3))
print('formula says:  slope', round(a, 4), ' intercept', round(b, 3))

They agree. That is worth pausing on: two completely different routes, one
algebraic, one a numerical search that knows nothing about correlation, arrive
at the same line.

`minimize` matters because it keeps working when there is no formula. Change the
error function, add a second predictor, fit a curve, and the algebra gets hard or
impossible while `minimize` carries on unchanged.

> **One caution.** `minimize` searches numerically, and it can struggle when the
> two quantities are on very different scales, if your x values all sit near
> 170, say, the intercept is being extrapolated a long way and the search can
> stop short of the true minimum. If `minimize` and the formula disagree,
> compare their RMSEs: the formula's is always the smaller one.


---

<a id='7'></a>
## **7. Residual plots**

A **residual** is what the line got wrong: actual minus predicted. Plot the
residuals against $x$ and you learn whether a line was the right shape.

**A good residual plot looks like nothing**, a formless band around zero. Any
pattern means the line is missing something.


In [ ]:
def residuals(t, x, y):
    return t.column(y) - fitted_all(t, x, y)

def fitted_all(t, x, y):
    return slope(t, x, y) * t.column(x) + intercept(t, x, y)

study_res = study.with_column('Residual', residuals(study, 'Hours', 'Mark'))
study_res.scatter('Hours', 'Residual')
plots.axhline(0, color='black', lw=1)
plots.title('Good: no pattern')
plots.show()

In [ ]:
# The same plot when a line is the wrong shape
bend = Table().with_columns('x', x, 'y', x ** 2 + np.random.normal(0, 0.6, len(x)))
bend_res = bend.with_column('Residual', residuals(bend, 'x', 'y'))
bend_res.scatter('x', 'Residual')
plots.axhline(0, color='black', lw=1)
plots.title('Bad: a clear curve -- a line was the wrong model')
plots.show()

The second plot's U-shape says the relationship bends and the line cannot follow
it. **You would not see that from $r$ or from RMSE alone**, the scatter and the
residual plot are what reveal it.

Two other patterns worth knowing: residuals that fan out as $x$ grows mean the
prediction is less reliable at one end, and a residual plot with an outlier far
from the rest usually means one point is dragging the whole line.


---

<a id='8'></a>
## **8. Inference: is the slope real?**

Here is the question everything has been building towards.

The slope you computed came from **a sample**. A different sample would give a
different slope. So: **could the true slope be zero?** If it could, the
relationship you are looking at might be nothing at all.

The arithmetic never tells you. Feed the regression formula pure noise and it
returns a slope regardless.


In [ ]:
# 40 points of pure noise -- no relationship whatsoever
np.random.seed(21)
noise = Table().with_columns('x', np.random.normal(0, 1, 40),
                             'y', np.random.normal(0, 1, 40))
noise.scatter('x', 'y', fit_line=True)
plots.title(f'r = {correlation(noise, "x", "y"):.3f}, '
            f'slope = {slope(noise, "x", "y"):.3f} -- from nothing')
plots.show()

The line tilts. It always does. So a slope on its own is not evidence.

The answer is Section 5's bootstrap, applied to a slope instead of a median.
**Resample the rows, refit the line, repeat.**


In [ ]:
def bootstrap_slopes(t, x, y, repetitions):
    """A bootstrap distribution of the regression slope."""
    slopes = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        slopes = np.append(slopes, slope(resample, x, y))
    return slopes

study_slopes = bootstrap_slopes(study, 'Hours', 'Mark', 1500)
Table().with_column('Bootstrap slope', study_slopes).hist()
plots.title('1500 bootstrap slopes -- mark on hours')
plots.show()

lo, hi = percentile(2.5, study_slopes), percentile(97.5, study_slopes)
print(f'95% interval for the true slope: [{lo:.4f}, {hi:.4f}]')
print('contains zero:', lo <= 0 <= hi)

Lab 6 Question 4.2 asks you to write this loop yourself rather than calling a
ready-made function. The four lines above are the shape it takes: start with an
empty array, resample, refit, append.

Zero is nowhere near that interval, so the relationship is real.

Now the same test on the noise.


In [ ]:
noise_slopes = bootstrap_slopes(noise, 'x', 'y', 1500)
lo2, hi2 = percentile(2.5, noise_slopes), percentile(97.5, noise_slopes)
print(f'95% interval for the noise slope: [{lo2:.3f}, {hi2:.3f}]')
print('contains zero:', lo2 <= 0 <= hi2)

It does. A flat line is entirely plausible for that data, so you should draw no
conclusion from the tilt you can see in the scatter.

**That is the whole of Chapter 16**, and it is why the bootstrap was worth three
Section's of build-up:

| The interval | Conclusion |
|---|---|
| excludes zero | the relationship is real at that confidence level |
| contains zero | a flat line is plausible; report nothing |

It is the same machinery as Section 5, on a different statistic. Nothing new, which
is the point.


---
## **Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** A scatter plot shows a clear curved pattern, and r is close to 0. Is there no relationship?

<details>
<summary><strong>Answer</strong></summary>

There may be a very strong relationship. <code>r</code> measures only the <strong>linear</strong> part of an association. A perfect U-shape can have r near zero. Always look at the plot before trusting r.

</details>

**Q2.** The lectures and textbook write `correlation(t, x, y)`; Labs 5 and 6 write `correlation(x, y)`. Which is right?

<details>
<summary><strong>Answer</strong></summary>

Both. They are Berkeley's own two conventions and compute the same number. The difference is only <strong>where the columns come out of the table</strong>: the lab version expects you to pull them out first, which is exactly what Lab 5's <code>predict(tbl, col1, col2)</code> does in its opening lines.

</details>

**Q3.** A tall father's son is usually shorter than him, and a short father's son taller. Does this mean heights are converging over generations?

<details>
<summary><strong>Answer</strong></summary>

No. This is <strong>regression to the mean</strong>, and it happens whenever <code>|r| &lt; 1</code>. In standard units the predicted y is <code>r</code> times the x, so a prediction is always closer to average than the x that produced it. The spread of heights is unchanged.

</details>

**Q4.** What should a residual plot look like if a straight line is a good fit, and what does a curve in it mean?

<details>
<summary><strong>Answer</strong></summary>

A <strong>formless horizontal band</strong> with no pattern. Any curve means the relationship is not linear and a straight line is the wrong model, however good r looked. Uneven vertical spread means the line is more reliable in some ranges than others.

</details>

**Q5.** What does `minimize` do, and why is it worth running when the formula already gives the answer?

<details>
<summary><strong>Answer</strong></summary>

It searches numerically for the slope and intercept with the smallest RMSE, knowing nothing about correlation or any formula. It lands on the same line as the algebra, which is how you know the formula is not an arbitrary recipe: it is the line with the least squared error.

</details>

**Q6.** Your bootstrapped 95% interval for the slope is [0.59, 0.82]. What do you conclude?

<details>
<summary><strong>Answer</strong></summary>

Zero is outside the interval, so at the 5% level you reject the hypothesis that the true slope is zero. The association is unlikely to be an accident of this particular sample. It does not establish that x causes y.

</details>

---

<a id='9'></a>
## **9. Quick reference**

### The functions, in order

```python
def standard_units(x):
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    return np.mean(standard_units(t.column(x)) * standard_units(t.column(y)))

def slope(t, x, y):
    return correlation(t, x, y) * np.std(t.column(y)) / np.std(t.column(x))

def intercept(t, x, y):
    return np.mean(t.column(y)) - slope(t, x, y) * np.mean(t.column(x))
```

### **Bootstrapping a slope**

```python
slopes = make_array()
for i in np.arange(1500):
    slopes = np.append(slopes, slope(t.sample(), x, y))
percentile(2.5, slopes), percentile(97.5, slopes)
```

### What each thing tells you

| | Answers |
|---|---|
| Scatter plot | is a line even sensible? |
| `r` | how tight is the linear association? |
| slope, intercept | what is the best line? |
| RMSE | how far off are the predictions? |
| Residual plot | is a line the right *shape*? |
| Bootstrap interval | **could the true slope be zero?** |

### **Things that catch people out**

| | |
|---|---|
| `r` near 0 | means no *linear* association, the relationship may still be exact |
| A slope you can see | appears in pure noise too |
| Predicting outside the data | the line was never fitted there |
| A patterned residual plot | the line is the wrong shape, whatever `r` says |
| Correlation | still not causation, however small the interval |

---

### **Textbook**

- [Chapter 15, Prediction](https://inferentialthinking.com/chapters/15/Prediction.html)
- [Chapter 15.2, The regression line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)
- [Chapter 15.3, The method of least squares](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)
- [Chapter 15.5, Visual diagnostics](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)
- [Chapter 16, Inference for Regression](https://inferentialthinking.com/chapters/16/Inference_for_Regression.html)
